Imports

In [1]:
import feedparser
import pandas as pd
from datetime import datetime, timedelta
import time
import sys
import requests
from bs4 import BeautifulSoup
import finviz
import os


Fraser Data Collection

In [2]:
def scrape_fraser_timeline(url):
    """Scrapes date, title, description, and link info from the timeline."""
    try:
        # 1. Download the HTML content
        response = requests.get(url)
        response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)

        # 2. Parse the HTML
        soup = BeautifulSoup(response.content, 'html.parser')

        # 3. Find the main container element
        # Your target container is class="timeline-events clusterize-scroll" and id="list-container"
        timeline_container = soup.find('div', id='list-container')
        
        if not timeline_container:
            print("Error: Main timeline container not found.")
            return []

        # 4. Find all individual event rows
        # The articles are inside <div class="row event-row active">
        event_rows = timeline_container.find_all('div', class_='event-row')

        data = []
        for row in event_rows:
            # 5. Extract data points using the specific classes
            
            # Date and Source/Title: <h2 class="list-item">
            header_element = row.find('h2', class_='list-item')
            header_text = header_element.text.strip() if header_element else 'N/A'
            
            # Description/Summary: <p class="list-item">
            summary_element = row.find('p', class_='list-item')
            summary = summary_element.text.strip() if summary_element else 'N/A'
            
            # Associated Link: <ul><li><a href="..." class="list-item">
            link_element = row.find('a', class_='list-item')
            
            link_title = link_element.text.strip() if link_element else 'N/A'
            link_url = link_element['href'] if link_element else 'N/A'
            
            # Prepend the base URL if the link is relative
            if link_url != 'N/A' and link_url.startswith('/'):
                link_url = 'https://fraser.stlouisfed.org' + link_url
            
            # Split the header into Date and Source
            if '|' in header_text:
                date, source = [part.strip() for part in header_text.split('|', 1)]
            else:
                date = header_text
                source = 'N/A'

            data.append({
                'Date': date,
                'Source': source,
                'Summary': summary,
                'Associated Link Title': link_title,
                'Associated Link URL': link_url
            })

        return data

    except requests.exceptions.RequestException as e:
        print(f"An error occurred during the request: {e}")
        return []
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return []
    
# Get Timeline Data from Fraser
# --- Execution ---
FINANCIAL_CRISIS_URL = "https://fraser.stlouisfed.org/timeline/financial-crisis"
fraser_financial_crisis_timeline_data = scrape_fraser_timeline(FINANCIAL_CRISIS_URL)
fraser_financial_crisis_timeline_df = pd.DataFrame(fraser_financial_crisis_timeline_data)
URL_COVID = "https://fraser.stlouisfed.org/timeline/covid-19-pandemic"
fraser_covid_timeline_data = scrape_fraser_timeline(URL_COVID)
fraser_covid_timeline_df = pd.DataFrame(fraser_covid_timeline_data)

# Save DataFrames to CSV files7
fraser_financial_crisis_timeline_df.to_csv('Data/fraser_financial_crisis_timeline.csv', index=False)    
fraser_covid_timeline_df.to_csv('Data/fraser_covid_timeline.csv', index=False)

Define Tickers

In [6]:
# 1. The "Too Big to Fail" (G-SIBs)
bank_tickers = ['JPM', 'BAC', 'C', 'HSBC', 'IDCBY', 'GS', 'BNPQY', 'UBS', 'ACGBY', 'BACHY', 'CICHY', 'BCS', 'MUFG', 'MS', 'WFC', 'BK', 'STT', 'DB', 'ING', 'SAN', 'RY', 'TD', 'SCBFF', 'SCGLY', 'MFG', 'SMFG', 'BCMXY', 'GCRLY', 'BPCE']
# 2. Major Financial Intermediaries (Non-Bank)
# A. Global Stock Exchanges
stock_exchange_tickers = ['ICE', 'NDAQ', 'CME', 'CBOE', 'LSEG']
# B. Largest Global Asset Managers
asset_manager_tickers = ['BLK', 'AMP', 'IVZ', 'BEN', 'TROW', 'BAM', 'BX']

Finviz Data Collection

In [7]:
# Defined list of Global Systemically Important Banks (G-SIBs) 
# and KBW Index members that work with Finviz
#bank_tickers = [
#   'JPM', 'BAC', 'WFC', 'C', 'GS', 'MS',  # US Giants
#   'HSBC', 'SAN', 'BCS', 'DB', 'UBS',    # European Leaders (ADRs)
#   'RY', 'TD'                             # Canadian Leaders
#]

def fetch_bank_news(ticker_list):
    compiled_news = {}
    
    for ticker in ticker_list:
        try:
            print(f"Fetching news for {ticker}...")
            # Using the working individual stock news function
            news = finviz.get_news(ticker)
    
            compiled_news[ticker] = news
            # Small sleep to avoid rate limiting
            time.sleep(0.5) 
        except Exception as e:
            print(f"Could not fetch {ticker}: {e}")
            
    return compiled_news

# Run the fetcher
finviz_news = fetch_bank_news(bank_tickers)
print(finviz_news)
finviz_news_df = pd.DataFrame(finviz_news)
finviz_news_df.to_csv('Data/finviz_news.csv', index=False)

Fetching news for JPM...
Fetching news for BAC...
Fetching news for C...
Fetching news for HSBC...
Fetching news for IDCBY...
Could not fetch IDCBY: 404 Client Error: Not Found for url: https://finviz.com/quote.ashx?t=IDCBY
Fetching news for GS...
Fetching news for BNPQY...
Could not fetch BNPQY: 404 Client Error: Not Found for url: https://finviz.com/quote.ashx?t=BNPQY
Fetching news for UBS...
Fetching news for ACGBY...
Could not fetch ACGBY: 404 Client Error: Not Found for url: https://finviz.com/quote.ashx?t=ACGBY
Fetching news for BACHY...
Could not fetch BACHY: 404 Client Error: Not Found for url: https://finviz.com/quote.ashx?t=BACHY
Fetching news for CICHY...
Could not fetch CICHY: 404 Client Error: Not Found for url: https://finviz.com/quote.ashx?t=CICHY
Fetching news for BCS...
Fetching news for MUFG...
Fetching news for MS...
Fetching news for WFC...
Fetching news for BK...
Fetching news for STT...
Fetching news for DB...
Fetching news for ING...
Fetching news for SAN...
Fetc

FINNHUB Data Collection

In [8]:
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv('FINNHUB_API_KEY')

def fetch_all_ticker_news(ticker_list):
    all_news_items = []
    to_date = datetime.now().strftime('%Y-%m-%d')
    from_date = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')

    for ticker in ticker_list:
        print(f"Fetching news for {ticker}...")
        url = f'https://finnhub.io/api/v1/company-news?symbol={ticker}&from={from_date}&to={to_date}&token={API_KEY}'
        
        response = requests.get(url)
        if response.status_code == 200:
            news_data = response.json()
            
            # Add the Ticker symbol to each news entry so the CSV is organized
            for item in news_data:
                item['ticker_symbol'] = ticker # Custom column
                all_news_items.append(item)
        
        # Respect the free tier limit (60 calls/min)
        time.sleep(1) 
        
    return all_news_items

# 1. Run the fetcher
all_news = fetch_all_ticker_news(bank_tickers)

# 2. Convert to DataFrame
df = pd.DataFrame(all_news)

# 3. Clean up the date (Finnhub uses Unix timestamps)
if not df.empty:
    df['date'] = pd.to_datetime(df['datetime'], unit='s')
    
    # 4. Save to CSV
    df.to_csv('Data/finnhub_news.csv', index=False)
    print(f"Successfully saved {len(df)} news items to Data/finnhub_news.csv")
else:
    print("No news found to save.")

Fetching news for JPM...
Fetching news for BAC...
Fetching news for C...
Fetching news for HSBC...
Fetching news for IDCBY...
Fetching news for GS...
Fetching news for BNPQY...
Fetching news for UBS...
Fetching news for ACGBY...
Fetching news for BACHY...
Fetching news for CICHY...
Fetching news for BCS...
Fetching news for MUFG...
Fetching news for MS...
Fetching news for WFC...
Fetching news for BK...
Fetching news for STT...
Fetching news for DB...
Fetching news for ING...
Fetching news for SAN...
Fetching news for RY...
Fetching news for TD...
Fetching news for SCBFF...
Fetching news for SCGLY...
Fetching news for MFG...
Fetching news for SMFG...
Fetching news for BCMXY...
Fetching news for GCRLY...
Fetching news for BPCE...
Successfully saved 5006 news items to Data/finnhub_news.csv
